# Laboratorio: Detección de outliers y estadística en Python

**Duración estimada:** 60–90 minutos  
**Objetivo:** practicar herramientas estadísticas y detección de outliers en Python, combinando enfoques univariados y multivariados, e incorporando intervalos de confianza y pruebas de hipótesis.

**Contenido:**
- Estadística descriptiva y medidas robustas.
- Outliers univariados: Z-score, IQR, **MAD** (z robusto).
- Outliers multivariados: **IsolationForest**.
- Intervalos de confianza para la media.
- Pruebas t de dos muestras y correlación (Pearson/Spearman).

**Indicaciones:** completa solo donde dice`#yourcodehere`. Hay *sanity checks* visibles y **tests ocultos** entre. Usa únicamente `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

In [ ]:
# === Setup (no modificar) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Tuple

from sklearn.ensemble import IsolationForest
from scipy import stats

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
np.set_printoptions(precision=3, suppress=True)
pd.set_option('display.precision', 3)

print('Librerías OK')
import sys, sklearn
print('Python', sys.version.split()[0], '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)

## Datos de trabajo (sintéticos con outliers)
Se generarán dos conjuntos de datos sintéticos:
- **A_univar:** distribución normal con outliers añadidos.
- **B_multivar:** dos variables correlacionadas con outliers multivariados añadidos.

In [ ]:
# Generación de datos (no modificar)
n = 400
A_core = rng.normal(loc=50, scale=10, size=n)
A_out = np.array([120, 115, 5, -10, 130, 111], dtype=float)
A_univar = np.concatenate([A_core, A_out])

m = 500
mean = np.array([0.0, 0.0])
cov = np.array([[1.0, 0.8], [0.8, 1.5]])
B_core = rng.multivariate_normal(mean, cov, size=m)
B_out = rng.uniform(low=[6, -9], high=[9, -6], size=(15,2))  # outliers alejados
B_multivar = np.vstack([B_core, B_out])

print('A_univar shape:', A_univar.shape)
print('B_multivar shape:', B_multivar.shape)

# Visualizaciones rápidas
plt.figure(); plt.hist(A_univar, bins=30); plt.title('A_univar: histograma'); plt.tight_layout(); plt.show()
plt.figure(); plt.boxplot(A_univar, vert=False); plt.title('A_univar: boxplot'); plt.tight_layout(); plt.show()
plt.figure(); plt.scatter(B_multivar[:,0], B_multivar[:,1], s=8); plt.title('B_multivar: dispersión'); plt.tight_layout(); plt.show()

## Ejercicio 1 — Resumen estadístico con medidas robustas
Implementa `summary_stats(x: np.ndarray) -> dict` que retorne:
```python
{'n': int, 'mean': float, 'median': float, 'std': float, 'mad': float}
```
- `std` se calcula con `ddof=1`
- `mad` es la *Median Absolute Deviation*

In [ ]:
def summary_stats(x: np.ndarray) -> dict:
    x = np.asarray(x, dtype=float)
    n = x.size
    mean = 0
    median = 0
    std = 0
    mad = 0
    # your code here
    raise NotImplementedError
    return {'n': int(n), 'mean': mean, 'median': median, 'std': std, 'mad': mad}

In [ ]:
summ = summary_stats(A_univar)
assert set(summ.keys()) == {'n','mean','median','std','mad'}
assert summ['n'] == A_univar.size
print('OK: resumen funciona')

## Ejercicio 2 — Outliers por Z-score
Implementa `zscore_outliers(x, threshold=3.0)` que retorne los **índices** donde `|z| > threshold`.

In [ ]:
def zscore_outliers(x: np.ndarray, threshold: float = 3.0) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    #Calcula mu,sd
    #mu = ...
    #sd = ...
    
    # your code here
    raise NotImplementedError
    if sd == 0 or np.isnan(sd):
        return np.array([], dtype=int)
    z = (x - mu) / sd
    return np.where(np.abs(z) > threshold)[0]


In [ ]:
idx_z = zscore_outliers(A_univar, 3.0)
print('Outliers Z-score:', len(idx_z))

## Ejercicio 3 — Outliers por IQR
Implementa `iqr_outliers(x, k=1.5)` aplicando la regla de Tukey.

In [ ]:
def iqr_outliers(x: np.ndarray, k: float = 1.5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    q1, q3 = 0,0
    iqr = 0
    low = 0
    high = 0
    # your code here
    raise NotImplementedError
    mask = (x < low) | (x > high)
    return np.where(mask)[0]

idx_iqr = iqr_outliers(A_univar)
print('Outliers IQR:', len(idx_iqr))

## Ejercicio 4 — Outliers por Modified Z-score (MAD)
Implementa `mad_outliers(x, threshold=3.5)` que devuelva los indices de los outliers identificados por el método MAD

In [ ]:
def mad_outliers(x: np.ndarray, threshold: float = 3.5) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    mz = x # reemplazar
    # your code here
    raise NotImplementedError
    if mad == 0 or np.isnan(mad):
        return np.array([], dtype=int)
    return np.where(mz > threshold)[0]
    
idx_mad = mad_outliers(A_univar)
print('Outliers MAD:', len(idx_mad))

## Ejercicio 5 — Outliers multivariados con IsolationForest (no evaluado)

En este ejercicio, implementaremos una función que detecta outliers con el método IsolationForest.

Isolation Forest detecta outliers buscando qué tan fácil es aislar cada punto del resto.
Si un punto es muy distinto, se separa rápidamente al dividir los datos; por eso se considera outlier.
Los puntos normales requieren más divisiones para aislarlos.

**Parámetro contamination**

El parámetro `contamination` indica qué proporción aproximada de los datos esperas que sean outliers.
Por ejemplo, usar 0.03 significa asumir que alrededor del 3% de los datos son anómalos.
Este valor es común cuando los outliers son pocos pero reales: no es tan alto como para detectar falsos positivos ni tan bajo como para perder anomalías importantes.

**Cuándo usar Isolation Forest**

Es especialmente útil cuando:
- Tienes datos con muchas variables y las relaciones entre ellas son complejas.
- Los datos no siguen una distribución normal, donde z-score o IQR fallan.
- Trabajas con datasets grandes, donde los métodos tradicionales no escalan bien.
- Quieres detectar outliers que solo son raros al combinar varias características, no solo mirando cada variable por separado.

In [ ]:
def isolation_forest_flags(X: np.ndarray, contamination: float = 0.03, random_state: int = RANDOM_STATE) -> np.ndarray:
    iso = IsolationForest(contamination=contamination, random_state=random_state)
    labels = iso.fit_predict(np.asarray(X, float))
    return labels == -1

flags = isolation_forest_flags(B_multivar, 0.03)
print('Proporción de outliers:', flags.mean())
print('Outliers en el dataset:', B_multivar[flags== True])

## Ejercicio 6 — Intervalo de confianza para la media

Implementemos una función que calcula el intervalo de confianza para la media

Primero obtiene la media y el error estándar, y luego construye el intervalo usando:

- Distribución t (si use_t=True, recomendado cuando el tamaño de muestra es pequeño o la varianza es desconocida).
- Distribución normal (si use_t=False).

El parámetro `alpha` controla el nivel de confianza (por defecto 0.05 → 95%).
La función devuelve dos valores: límite inferior y límite superior del intervalo.

In [ ]:
def ci_mean(x: np.ndarray, alpha: float = 0.05, use_t: bool = True) -> Tuple[float, float]:
    x = np.asarray(x, float)
    n = x.size
    if n < 2:
        return float('nan'), float('nan')
    low = 0
    high = 0
    # your code here
    raise NotImplementedError
    return low, high

lo, hi = ci_mean(A_core)
print('IC95%:', lo, hi)

## Ejercicio 7 — Prueba t de dos muestras (no evaluado)

Esta función realiza un t-test de dos muestras para comparar si dos grupos (x y y) tienen medias significativamente diferentes.

- Usa `stats.ttest_ind` para obtener la estadística t y el p-value.
- Permite elegir si se asume o no igualdad de varianzas entre los grupos (equal_var).
- El parámetro `alternative` permite hacer un test:

- -- two-sided (por defecto): detecta diferencias en cualquier dirección.
- -- greater: prueba si la media de x es mayor que la de y.
- -- less: prueba si la media de x es menor que la de y.

La función devuelve la estadística t y el p-value ajustado según el tipo de prueba.

In [ ]:
def t_test_two_sample(x: np.ndarray, y: np.ndarray, equal_var: bool = False, alternative: str = 'two-sided') -> Tuple[float, float]:
    res = stats.ttest_ind(x, y, equal_var=equal_var, alternative='two-sided')
    t_stat, p_two = float(res.statistic), float(res.pvalue)
    alt = alternative.lower()
    if alt == 'two-sided':
        return t_stat, p_two
    p_one = p_two / 2
    if alt == 'greater':
        p = p_one if t_stat > 0 else 1 - p_one
    elif alt == 'less':
        p = p_one if t_stat < 0 else 1 - p_one
    else:
        raise ValueError('alternative debe ser two-sided | less | greater')
    return t_stat, float(p)

x1 = rng.normal(0,1,60)
x2 = rng.normal(0.5,1,55)
print(t_test_two_sample(x1, x2))

### Cómo interpretar los resultados del t-test

La función devuelve dos valores:

`t_stat` → la estadística t

`p-value` → la probabilidad de observar una diferencia como la vista (o mayor) si en realidad no hubiera diferencia entre medias.

1. Interpretación del p-value

El punto clave es el `p-value`:

Si `p < 0.05` (nivel típico de significancia):
→ Hay evidencia estadística de que las medias son diferentes (o en la dirección especificada si es “greater” o “less”).
→ Se rechaza la hipótesis nula de igualdad de medias.

Si `p ≥ 0.05`:
→ No hay suficiente evidencia para afirmar que las medias son distintas.
→ No se rechaza la hipótesis nula.

**Importante: no significa que las medias sean iguales, solo que no hay evidencia suficiente para afirmar lo contrario.**

2. Interpretación de la estadística t

El valor de `t` indica:

Qué tan grande es la diferencia entre medias comparada con la variabilidad de los datos.

Si `t` es positivo → la media de x es mayor que la de y.

Si `t` es negativo → la media de x es menor que la de y.

Pero, por sí solo, `t` no se interpreta en términos de significancia; eso viene del p-value.

## Ejercicio 8 — Correlación Pearson/Spearman (no evaluado)

Esta función calcula la correlación entre dos variables (x y y) y realiza un test estadístico para evaluar si esa correlación es significativa.

Puede usar dos métodos:

`pearson` → mide correlación lineal (requiere relación aproximadamente lineal y variables continuas).

`spearman` → mide correlación por rangos (detecta relaciones monótonas, no necesita linealidad).

La función devuelve:

`r` → el coeficiente de correlación

`p-value` → la probabilidad de observar una correlación como esa si, en realidad, no hubiera relación entre las variables

In [ ]:
def corr_test(x: np.ndarray, y: np.ndarray, method: str='pearson') -> Tuple[float, float]:
    method = method.lower()
    if method == 'pearson':
        r, p = stats.pearsonr(x, y)
    elif method == 'spearman':
        r, p = stats.spearmanr(x, y)
    else:
        raise ValueError('method debe ser pearson | spearman')
    return float(r), float(p)

r_p, p_p = corr_test(B_multivar[:,0], B_multivar[:,1])
print(r_p, p_p)

Cómo interpretar los resultados
1. Interpretación de r (correlación)

Valores típicos (cercanos a):

`r ≈ 1` → correlación positiva fuerte

`r ≈ -1` → correlación negativa fuerte

`r ≈ 0` → poca o ninguna correlación

Indica qué tan relacionado está un cambio en x con un cambio en y.

2. Interpretación del `p-value`

`p < 0.05` → la correlación observada es estadísticamente significativa.
Es decir, es improbable que el valor de r se deba al azar.

`p ≥ 0.05` → no hay evidencia suficiente para afirmar que las variables están correlacionadas.

## Cierre
Este laboratorio mostró cómo combinar herramientas de detección de anomalías y estadística inferencial para obtener una visión más completa y confiable de los datos.
Por un lado, comparamos métodos clásicos para identificar outliers (Z-score, IQR y MAD), entendiendo sus diferentes niveles de robustez. Luego incorporamos Isolation Forest, capaz de detectar anomalías que solo emergen cuando se consideran varias variables a la vez, algo que los métodos univariados no pueden capturar.

En la parte inferencial, trabajamos con intervalos de confianza, pruebas t y análisis de correlación, que permiten no solo describir los datos, sino evaluar hipótesis y relaciones de manera rigurosa.
Al integrar estas herramientas, el análisis se vuelve más sólido: primero se limpia y entiende la estructura de los datos, y luego se aplican pruebas estadísticas con mayor confianza y menor riesgo de errores causados por valores atípicos.

En conjunto, el laboratorio muestra que un buen análisis de datos combina técnicas de detección de anomalías con métodos inferenciales, logrando conclusiones más precisas, interpretables y confiables.

**Ejecuta Kernel → Restart & Run All antes de entregar.**